# ViMedAQA baseline RAG (VinaLLaMA + BGE-M3)
Settings: **Accelerator = GPU T4**, **Internet = On**. Run cells top to bottom.
If you saved the index as a Kaggle Dataset named `vimedrag-index`, add it via *Add Input* and skip the build cell.

In [ ]:
!nvidia-smi

In [ ]:
REPO_URL = "https://github.com/YOUR_GITHUB_USER/vimedrag.git"  # <-- sửa thành repo của bạn
import os
WORK = "/kaggle/working/vimedrag"
if os.path.isdir(WORK):
    !git -C {WORK} pull
else:
    !git clone {REPO_URL} {WORK}
os.chdir(WORK)
!git log --oneline -1

In [ ]:
!pip install -q -U bitsandbytes bert-score accelerate

In [ ]:
import torch, transformers, bitsandbytes; print(torch.__version__, transformers.__version__, bitsandbytes.__version__)

In [ ]:
!python -m pytest -q

In [ ]:
import glob
_hits = sorted(glob.glob("/kaggle/input/**/meta.json", recursive=True))
INDEX_DIR = os.path.dirname(_hits[0]) if _hits else "/kaggle/working/index"
RUNS_DIR = "/kaggle/working/runs"
print(INDEX_DIR)

In [ ]:
# Build index (~10-20 phút). Tự bỏ qua nếu đã có meta.json
if os.path.exists(os.path.join(INDEX_DIR, "meta.json")):
    print("Index đã có, bỏ qua build:", INDEX_DIR)
else:
    print("Chưa có index, đang build:", INDEX_DIR)
    !python scripts/build_index.py --index-dir {INDEX_DIR}

In [ ]:
# Smoke test (~5 phút): 20 câu, kiểm tra retrieval + prompt + stop token
!python scripts/run_eval.py --mode rag --n-eval 20 --no-bertscore --exp-name smoke_rag --index-dir {INDEX_DIR} --runs-dir {RUNS_DIR}
!head -n 3 {RUNS_DIR}/smoke_rag/predictions.jsonl

In [ ]:
# E0: closed-book
!python scripts/run_eval.py --mode closed --index-dir {INDEX_DIR} --runs-dir {RUNS_DIR}

In [ ]:
# E1: RAG top-3
!python scripts/run_eval.py --mode rag --top-k 3 --index-dir {INDEX_DIR} --runs-dir {RUNS_DIR}

In [ ]:
# E2: oracle context
!python scripts/run_eval.py --mode oracle --index-dir {INDEX_DIR} --runs-dir {RUNS_DIR}

In [ ]:
!python scripts/summarize.py {RUNS_DIR}
!head -n 3 {RUNS_DIR}/rag_k3_vinallama7b/predictions.jsonl

## Sau khi chạy
- Lưu index để lần sau khỏi encode lại: *Save Version* → Output, rồi tạo Dataset `vimedrag-index` từ thư mục `/kaggle/working/index`.
- Nếu session bị ngắt: chạy lại cùng cell, pipeline tự bỏ qua câu đã xong (resume). Lưu ý `/kaggle/working` mất khi mở session mới nếu chưa Save Version.
- Gửi lại: `runs/*/metrics.json` + vài dòng `predictions.jsonl`.